# GRAT walkthrough: Oak Valley sample data

GRAT models how water available for recharge (WAFR) is allocated, day by day, to canals, basins and farm fields, and reports how much of it reaches groundwater.

This notebook runs the full workflow end to end against **Oak Valley**, a synthetic sample dataset that ships with the `grat` package (it's included whether you `pip install grat` or clone the repo). It is not real hydrologic, crop or parcel data.

Steps:

0. Install `grat` (skip if it's already installed).
1. Load the sample data (shapefile + csvs).
2. **Check it for problems before running anything**, with `grat.validate_grat_inputs`.
3. Choose the scenario - the same choices as the GRAT web app's *Select Projects to Assess* and *Set Priority Criteria* screens.
4. Build the model's site objects for that scenario.
5. Run the model and produce reports.
6. Read a report back as a dataframe.

## 0. Install grat

Installs `grat` from PyPI, along with the Oak Valley sample data that ships with it. If `grat` is already installed, this upgrades it to the latest release if needed.

In [ ]:
%pip install --quiet --upgrade grat

## 1. Load the sample data

In [ ]:
from importlib.resources import files
from pathlib import Path

import pandas as pd

OAK_VALLEY = Path(str(files("grat.datasets") / "OakValley"))
list(OAK_VALLEY.iterdir())

`load_sites` reads the attribute table of any vector file GDAL can read (shapefile, file geodatabase, geopackage, geojson) or a plain csv - geometry is never read, so no GIS library beyond `pyogrio` is needed.

The sites layer's raw field names (`GRATID`, `R_TYPE`, ...) rarely match what the model classes expect (`grat_id`, `recharge_type`, ...), so `attribute_renamer` maps one to the other. This is the mapping for Oak Valley's shapefile; bringing your own data means building the equivalent dictionary for your own field names.

In [ ]:
from grat.loaders.get_sites import load_sites

ATTRIBUTE_RENAMER = {
    "GRATID": "grat_id",
    "R_TYPE": "recharge_type",
    "ACRES": "acres",
    "CROP_OFR": "crop",
    "CROP_IL": "in_lieu_crop",
    "WAFRSOURCE": "wafr_sources_primary",
    "WAFRSOUR_1": "wafr_sources_secondary",
    "BOTTLENECK": "bottleneck_ids",
    "SAGBI_CLAS": "soil_type",
    "PRECIP_IDX": "precip_grid",
    "BASIN_CAP": "max_capacity",
    "BASIN_PERC": "perc_rate",
    "CFS_LIMIT": "cfs_limit",
    "RSI": "rsi",
    "RMA": "rma",
    "ET_Region": "region_number",
    "DISTRICT": "district",
    "SUBBASIN": "subbasin",
    "WATERSHED": "watershed",
    "AGENCY": "agency",
    "PROJECTS": "projects",
}

site_rows = load_sites(OAK_VALLEY / "oakvalley.shp", attribute_renamer=ATTRIBUTE_RENAMER)
len(site_rows), site_rows[0]

Canals, bottlenecks, WAFR, precipitation and the crop compatibility calendars each come from their own csv.

In [ ]:
from grat.loaders.get_bottlenecks import load_bottlenecks
from grat.loaders.get_canals import load_canals
from grat.loaders.get_ccc import get_crop_calendar, get_il_crop_calendar
from grat.loaders.get_precip import get_precip
from grat.loaders.get_wafr import load_wafr

canals = load_canals(OAK_VALLEY / "canals.csv")
bottlenecks = load_bottlenecks(OAK_VALLEY / "bottlenecks.csv")
wafr = load_wafr(OAK_VALLEY / "WAFR_optimistic.csv")
precip = get_precip(OAK_VALLEY / "Precip_optimistic.csv")
crop_calendar = get_crop_calendar(OAK_VALLEY / "CCC_ofr.csv")
crop_calendar_in_lieu = get_il_crop_calendar(OAK_VALLEY / "CCC_inlieu.csv")

len(canals), len(bottlenecks), sorted(wafr[2000])

## 2. Check the data before running the model

`validate_grat_inputs` runs data validation checks - missing or duplicate ids, non-numeric or negative values, crop/soil/WAFR-source/bottleneck references that don't match the other inputs, inconsistent day counts.

In [ ]:
from grat.validation import validate_grat_inputs

report = validate_grat_inputs(
    site_rows,
    wafr,
    bottleneck_rows=pd.read_csv(OAK_VALLEY / "bottlenecks.csv").to_dict("records"),
    crop_calendar=crop_calendar,
    crop_calendar_in_lieu=crop_calendar_in_lieu,
    precip=precip,
)
report

In this example `report.ok` is `True`, but the warning still matters. In the sample data, 32 fields list `"Wheat"` as their in-lieu crop with no on-farm crop, and the in-lieu crop calendar has no Wheat row. The model runs anyway, but it silently drops those fields: step 4 builds only 653 of the 685 sites.

In [ ]:
report.warnings

## 3. Choose the scenario

**Select Projects to Assess**
- **Recharge types** - turn recharge basins, on-farm recharge and in-lieu recharge on or off.
- **Build new conveyance** - Oak Valley has two planned projects, `Yolo Creek Extension` and `Lateral M`. They're off by default, so the fields that depend on them are left out; turn one on to include its fields.

**Set Priority Criteria**
- **Groundwater depth** - `None` includes every field; `20` or `50` includes only fields deeper than that many feet to groundwater. *Oak Valley has no depth-to-groundwater field, so leave this at `None` for the sample data - a limit would leave out every field. With your own data, map your depth field to `depth_to_groundwater` in `ATTRIBUTE_RENAMER`.*
- **Recharge management areas** - on-farm fields are ordered by RMA rank (rank 1 first) then recharge suitability (RSI) score when enabled, or by RSI score when disabled. In-lieu fields are ordered by groundwater change (lowest first), with RMA rank first when enabled.
- **Land repurposing** - the percentage (0-100) of each agency's acreage to convert to MLRP land. Oak Valley has one agency, `Sacramento GSA`. *Oak Valley has no MLRP score field, so the fields converted are simply the first ones in the data. With your own data, map your MLRP score field to `mlrp_score`; the lowest scores are converted first.*

In [ ]:
# Select Projects to Assess
RECHARGE_BASINS = True
ON_FARM_RECHARGE = True
IN_LIEU_RECHARGE = True

BUILD_NEW_CONVEYANCE = {
    "Yolo Creek Extension": False,
    "Lateral M": False,
}

# Set Priority Criteria
GROUNDWATER_DEPTH_FT = None  # None, 20 or 50
RECHARGE_MANAGEMENT_AREAS = True

LAND_REPURPOSING_PERCENT = {
    "Sacramento GSA": 0,
}

## 4. Build the sites for the scenario

`build_sites` turns the raw dictionaries into `Farm`/`Basin` objects, and `drop_invalid_sites` removes the ones whose crop, in-lieu crop or soil type don't match the crop compatibility calendar. The cell after it applies the settings from step 3; re-run both whenever you change a setting.

In [ ]:
from grat.loaders.get_ccc import ccc_valid_crops, ccc_valid_soil_types
from grat.loaders.get_sites import build_sites, drop_invalid_sites

all_sites = build_sites(site_rows)
all_sites = drop_invalid_sites(
    all_sites,
    valid_crop_list=ccc_valid_crops(crop_calendar),
    valid_il_crop_list=ccc_valid_crops(crop_calendar_in_lieu),
    valid_soil_types=ccc_valid_soil_types(crop_calendar),
)
len(all_sites)

In [ ]:
from grat.loaders.get_ccc import add_mlrp_to_ccc
from grat.model.sorters import mlrp_sorter, sort_sites

# Projects: leave out fields that depend on a project that isn't being built
sites = [site for site in all_sites if site.projects is None or BUILD_NEW_CONVEYANCE.get(site.projects)]

# Recharge Basins
if not RECHARGE_BASINS:
    sites = [site for site in sites if site.recharge_type != "Basin"]

# Groundwater Depths: keep basins, and fields deeper than the limit
if GROUNDWATER_DEPTH_FT is not None:
    sites = [
        site
        for site in sites
        if site.recharge_type == "Basin" or (site.depth_to_groundwater or 0) > GROUNDWATER_DEPTH_FT
    ]

# Land Repurposing: convert a share of each agency's fields to the MLRP crop
crop_calendar = add_mlrp_to_ccc(crop_calendar)
fields = [site for site in sites if site.recharge_type == "OFR"]
mlrp_sorter(fields, {agency: percent / 100 for agency, percent in LAND_REPURPOSING_PERCENT.items()})

# Recharge Management Areas: set the order the sites are given water
sites = sort_sites(canals + sites, sort_rule="RMA" if RECHARGE_MANAGEMENT_AREAS else None)

pd.Series([site.recharge_type for site in sites]).value_counts()

## 5. Run the model and produce reports

`run_grat` starts the model and writes the requested reports to an output folder (zipped by default). The sample data covers water years 1969-2018; this runs 2010-2012 so the notebook finishes quickly. Leave `years` out to run every year in the WAFR data.

`grat.reports.catalog.ALL_REPORTS` lists every report name that can be requested.

In [ ]:
import tempfile

from grat import run_grat

result = run_grat(
    sites,
    wafr,
    bottlenecks,
    crop_calendar,
    precip,
    crop_calendar_in_lieu=crop_calendar_in_lieu,
    use_ofr=ON_FARM_RECHARGE,
    use_in_lieu=IN_LIEU_RECHARGE,
    rma_sort=RECHARGE_MANAGEMENT_AREAS,
    years=range(2010, 2013),  # water years 2010-2012; leave out to run every year
    reports=["annual_summary_recharge", "acres_available_and_used", "fields_repurposed"],
    output_parent=tempfile.mkdtemp(prefix="grat_oak_valley_"),
)

result["reports"], result["zip_path"]

## 6. Load output reports as dataframes

Anything `run_grat` wrote to csv can be read back as a dataframe with `read_report`, without re-running the model.

In [ ]:
from grat import read_report

read_report(result["output_folder"], "annual_summary_recharge")